# Elektrische Flussprofile und Exportvollständigkeit

Der Workflow verwendet den konfigurierten Export **Aspern F7**, die vorhandenen Scenario-Zeitreihen und `pexl.plot.electricity_balance_facets`. Installation im Repository: `python -m pip install -e ".[publication]"`.

Die Darstellung umfasst vorhandene PV- und konventionelle Netzflüsse. Fehlende Verbraucher-, Speicher- oder signalgesteuerte Netzkanäle werden weder ergänzt noch als null interpretiert. Deshalb werden Teilflussprofile und keine geschlossene Verbrauchsbilanz ausgewiesen.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "website/_examples.yml").exists()
sys.path.insert(0, str(ROOT / "scripts"))

In [ ]:
from website_examples import configuration, load_entry, hourly_frame, flow_inventory, dynamic_figures
config = configuration()
scenario = load_entry(config["dynamic_example"]).scenarios[0]
frame = hourly_frame(scenario)
print(scenario.column_name, frame.index.min(), frame.index.max(), len(frame))
flow_inventory(scenario)

## Saisonale mittlere Tagesprofile

Die gemeinsame Implementierung verwendet `add_season`, gruppiert nach Jahreszeit und Tagesstunde und übergibt die Mittelwerte an `electricity_balance_facets`. Der Renderer selbst aggregiert nicht. Die stündlichen Energiemengen in Wh/m² NGF entsprechen erst nach Division durch den geprüften Zeitschritt von 1 h mittleren Leistungen in W/m² NGF.

Konventioneller Netzbezug und lokale PV-Nutzung erscheinen positiv, PV-Einspeisung negativ. Die Differenz dieser ausgewählten Beiträge ist kein Bilanzresiduum des gesamten Quartiers.

In [ ]:
figures = dynamic_figures(scenario)
figures["f7_seasons"].show()

## Chronologische Stundenwerte

Die Schaltflächen wählen konkrete Jänner- und Juliwochen. Zoom und Bereichsauswahl arbeiten auf den vollständigen 8.760 Stundenwerten. Ein Saisonmittel ist davon getrennt zu interpretieren.

In [ ]:
figures["f7_hourly"].show()

## Reproduzierbarer Export

Die Datenherkunft folgt demselben Format wie bei den Jahresbilanzen. Es wird keine Excel-Simulation gestartet.

In [ ]:
from pexl.plot.export import export_figure
from website_examples import provenance
output = ROOT / "notebooks/local_exports/electricity_profiles"
for name, figure in figures.items():
    export_figure(figure, output / f"{name}.html", provenance=provenance(
        [config["dynamic_example"]],
        "Ausgewählte elektrische Flüsse; keine geschlossene Verbrauchsbilanz. Wh/m² je 1 h / 1 h = W/m² NGF."))
print(output)

## Vorbereitung der vollständigen Zeitreihenbilanz

Der Kanalbestand kann für alle konfigurierten Beispiele geprüft werden. Die gemeinsame Prüffunktion unterscheidet fehlende Kanäle von exportierten Nullreihen. Zu untersuchen sind insbesondere konventioneller und signalgesteuerter Netzbezug, PV-Verwendung, Speicherbeladung/-entladung, EV-Ladung/-Rückspeisung und die Bilanzierung von Verlusten. Keine Verbrauchsreihe wird allein aus einer Differenz vorhandener Quellflüsse rekonstruiert.

In [ ]:
for entry in config["examples"]:
    current = load_entry(entry).scenarios[0]
    audit = flow_inventory(current)
    print(current.column_name, "vorhanden:", int(audit["Exportiert"].sum()), "von", len(audit))